In [ ]:
import pandas as pd
import geopandas as gpd

import sys
sys.path.append('..')
from helpers import (
    get_county2zone,
    calculate_intersection,
    get_bulk_power_sales,
    get_baseline_load_profiles
)

In [3]:
baseline_load_profiles = get_baseline_load_profiles()

In [ ]:
eia_code_total_load_map = {}
for year in range(2016, 2024):
    eia_code_total_load_map[year] = {}
    for eia_code, df in baseline_load_profiles.items():
        eia_code_total_load_map[year][eia_code] = df.loc[df.index.year == year].sum()

In [5]:
county2zone = get_county2zone(2023)
county2zone['state_fips'] = county2zone['FIPS'].str.split('p').str[1].str[:2]

In [ ]:
eia_zones = gpd.read_file('../data/shapefiles/bas_and_subbas.gpkg')

In [7]:
ct_pops = pd.read_csv('../data/DECENNIALDHC2020.P1_2025-08-20T004859/DECENNIALDHC2020.P1-Data.csv')
ct_pops['GEOID'] = ct_pops['GEO_ID'].str.split('US').str[1]
ct_pops = ct_pops.rename(columns={'P1_001N': 'population'})
ct_pops = ct_pops[['GEOID', 'population']]

In [8]:
all_state_cts = []
for state_fips in county2zone['state_fips'].unique().tolist():
    if state_fips == '09':
        continue

    state_cts = (
        gpd.read_file(f'../data/shapefiles/census_tracts/tl_2022_{state_fips}_tract')
        .to_crs(eia_zones.crs)
    )
    assert len(state_cts.merge(ct_pops, on='GEOID')) == len(state_cts)

    all_state_cts.append(state_cts)

state_cts = pd.concat(all_state_cts, ignore_index=True)
state_cts = state_cts.merge(ct_pops, on='GEOID')
state_cts['population_density'] = state_cts['population'].astype(float) / state_cts['ALAND']

In [9]:
zone_ct_intersects = calculate_intersection(eia_zones, state_cts, ['EIAcode', 'GEOID'])
zone_ct_intersects = zone_ct_intersects.loc[zone_ct_intersects.EIAcode != '4004'].copy()
zone_ct_intersects['FIPS'] = (
    'p' + zone_ct_intersects['STATEFP'].astype(str) + zone_ct_intersects['COUNTYFP'].astype(str)
)

In [10]:
zone_ct_intersects_init = zone_ct_intersects.copy()

In [11]:
county_load_percentage_by_year = {}

for load_year in range(2016, 2024):
    county2zone = get_county2zone(load_year)
    county_population_map = dict(zip(county2zone['FIPS'], county2zone['population']))

    zone_ct_intersects = zone_ct_intersects_init.copy()
    zone_ct_intersects['county_population'] = zone_ct_intersects['FIPS'].map(county_population_map)
    zone_ct_intersects['population'] = (
        zone_ct_intersects['population_density'] * zone_ct_intersects['geometry'].area
    )
    zone_ct_intersects['population'] = (
        zone_ct_intersects['population']
        / zone_ct_intersects.groupby('FIPS')['population'].transform('sum')
        * zone_ct_intersects['county_population']
    )
    zone_ct_intersects['percent_of_zone_population'] = (
        zone_ct_intersects['population']
        / zone_ct_intersects.groupby('EIAcode')['population'].transform('sum')
    )
    zone_ct_intersects['zonal_load'] = zone_ct_intersects['EIAcode'].map(eia_code_total_load_map[load_year])
    zone_ct_intersects['load'] = (
        zone_ct_intersects['zonal_load'] * zone_ct_intersects['percent_of_zone_population']
    )

    county_load = zone_ct_intersects.groupby(['FIPS', 'EIAcode'], as_index=False)['load'].sum()
    county_load = county_load.merge(county2zone[['FIPS', 'state']])
    county_load['percent_of_county_load_served'] = (
        county_load['load'] / county_load.groupby('FIPS')['load'].transform('sum')
    )
    county_load = (
        county_load.set_index(['FIPS', 'EIAcode'])
        ['percent_of_county_load_served']
        .sort_index()
    )
    for _, row in county2zone.loc[county2zone.state == 'CT'].iterrows():
        county_load.loc[(row['FIPS'], '4004')] = 1

    county_load_percentage_by_year[load_year] = county_load

In [12]:
county_load_percentages = pd.concat(county_load_percentage_by_year, axis=1)
assert county2zone.loc[~county2zone.FIPS.isin(county_load_percentages.index.get_level_values('FIPS'))].empty

In [13]:
county_load_percentages.to_csv('../data/county_zone_load_percentage.csv')

#### County load magnitudes for technical validation

In [14]:
county_load_by_year = {}

for load_year in range(2016, 2024):
    bulk_power_sales = get_bulk_power_sales(load_year)
    statewide_sales = (
        bulk_power_sales.groupby('State', as_index=False)
        ['Megawatthours']
        .sum(numeric_only=True)
        .rename(columns={'State': 'state', 'Megawatthours': 'value'})
        .set_index('state')
        ['value']
    )
    
    county2zone = get_county2zone(load_year)
    county_population_map = dict(zip(county2zone['FIPS'], county2zone['population']))

    zone_ct_intersects = zone_ct_intersects_init.copy()
    zone_ct_intersects['county_population'] = zone_ct_intersects['FIPS'].map(county_population_map)
    zone_ct_intersects['population'] = (
        zone_ct_intersects['population_density'] * zone_ct_intersects['geometry'].area
    )
    zone_ct_intersects['population'] = (
        zone_ct_intersects['population']
        / zone_ct_intersects.groupby('FIPS')['population'].transform('sum')
        * zone_ct_intersects['county_population']
    )
    zone_ct_intersects['percent_of_zone_population'] = (
        zone_ct_intersects['population']
        / zone_ct_intersects.groupby('EIAcode')['population'].transform('sum')
    )
    zone_ct_intersects['zonal_load'] = zone_ct_intersects['EIAcode'].map(eia_code_total_load_map[load_year])
    zone_ct_intersects['load'] = (
        zone_ct_intersects['zonal_load'] * zone_ct_intersects['percent_of_zone_population']
    )

    county_load = zone_ct_intersects.groupby(['FIPS', 'EIAcode'], as_index=False)['load'].sum()
    county_load = county_load.merge(county2zone[['FIPS', 'state']])
    county_load['load'] = (
        county_load['load']
        / county_load.groupby('state')['load'].transform('sum')
        * county_load['state'].map(statewide_sales)
    )
    county_load = (
        county_load.set_index(['FIPS'])
        ['load']
        .sort_index()
    )

    county_load_by_year[load_year] = county_load

In [15]:
county_load_magnitudes = pd.concat(county_load_by_year, axis=1)
county_load_magnitudes = county_load_magnitudes.groupby(county_load_magnitudes.index).sum()

In [19]:
county_load_magnitudes.to_csv('../data/county_load_magnitudes.csv')